# AIPerf Use Case 8 — Profiling Ranking / Reranking Models

Profile reranker endpoints with AIPerf. A reranking request pairs one **query** with N candidate **passages**; the model returns a relevance score per passage. AIPerf can generate that workload synthetically — with control over passages-per-request and query/passage token lengths — or replay a custom JSONL dataset.

Based on the NVIDIA tutorial [Profiling Ranking Models with AIPerf](https://docs.nvidia.com/aiperf/dev/tutorials/model-endpoint-guides/profile-ranking-models-with-ai-perf). These use-case notebooks demonstrate AIPerf capabilities directly; the repo's per-scenario bash scripts remain the source of truth for the Model Selection / Sizing suites.

## Table of contents

1. [Overview](#1-overview)
2. [Requirements and server setup](#2-requirements-and-server-setup)
3. [Configuration](#3-configuration)
4. [Test run — synthetic](#4-test-run--synthetic)
5. [Test run — custom input file](#5-test-run--custom-input-file)
6. [Results analysis](#6-results-analysis)

## 1. Overview

AIPerf supports two reranking API shapes, selected with `--endpoint-type`:

| `--endpoint-type` | API shape | Typical server |
|---|---|---|
| `hf_tei_rankings` | Hugging Face TEI `/rerank` | `text-embeddings-inference` container |
| `cohere_rankings` | Cohere Re-Rank `/v1/rerank` | vLLM with `--runner pooling` |

As with embeddings (UC7), rerankers respond with a single non-streaming payload, so there are **no token-level metrics** (no TTFT/ITL, no `--streaming`) — the story is **request latency** and **request throughput**, as a function of the workload shape you set with the `--rankings-*` flags.

In a RAG pipeline the reranker sits between retrieval and generation, so its latency at your retriever's top-k adds directly to end-to-end time-to-first-token. Profile at the passage count you actually rerank.

## 2. Requirements and server setup

- `aiperf` CLI — the setup cell below installs it via pip if it isn't already on `PATH`. Pin whatever version you use (repo convention: record the AIPerf version per run).
- One of the two server options below (or any endpoint speaking the same API shape).
- Notebook Python deps: `pip install -r notebooks/requirements.txt` (jupyter, pandas, matplotlib).

**Option A — Hugging Face TEI (`--endpoint-type hf_tei_rankings`):**

```bash
docker run --gpus all --rm -it -p 8080:80 \
  ghcr.io/huggingface/text-embeddings-inference:latest \
  --model-id BAAI/bge-reranker-base --port 80
```

Smoke test:

```bash
curl -s http://localhost:8080/rerank \
  -H "Content-Type: application/json" \
  -d '{"query": "What is AI?", "texts": ["AI is artificial intelligence.", "Bananas are yellow."]}'
```

**Option B — vLLM serving the Cohere Re-Rank API (`--endpoint-type cohere_rankings`):**

```bash
docker run --gpus all -p 8080:8000 -e HF_TOKEN=<your token> \
  vllm/vllm-openai:latest \
  --model BAAI/bge-reranker-v2-m3 --runner pooling
```

Smoke test:

```bash
curl -s http://localhost:8080/v1/rerank \
  -H "Content-Type: application/json" \
  -d '{"query": "What is AI?", "documents": ["Artificial intelligence overview", "Bananas are yellow"]}'
```

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)
print(f"Repo root: {REPO_ROOT}")

if shutil.which("aiperf") is None:
    print("aiperf CLI not found — installing into this kernel's environment ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "aiperf"], check=True)

aiperf_path = shutil.which("aiperf")
assert aiperf_path, "aiperf still not found after install — restart the kernel and rerun this cell."
print(f"aiperf found at: {aiperf_path}")
version = subprocess.run(["aiperf", "--version"], capture_output=True, text=True)
print((version.stdout or version.stderr).strip())


## 3. Configuration

| Flag | Meaning |
|---|---|
| `--endpoint-type hf_tei_rankings` / `cohere_rankings` | Which reranking API shape to speak |
| `--rankings-passages-mean` / `--rankings-passages-stddev` | Passages per request |
| `--rankings-passages-prompt-token-mean` / `-stddev` | Token length of each passage |
| `--rankings-query-prompt-token-mean` / `-stddev` | Token length of the query |
| `--input-file` + `--custom-dataset-type single_turn` | Replay your own query/passages records |

**Constraint** (documented in the tutorial): the `--rankings-*` token options cannot be combined with `--prompt-input-tokens-mean` or its related synthetic-prompt flags — rankings workloads are shaped exclusively by the `--rankings-*` flags.

Set `BACKEND`, `MODEL`, and `URL`, then run. Defaults match the tutorial: 10 requests, ~5 passages of ~32 tokens per request, ~16-token queries.

In [ ]:
# ---- Required ----------------------------------------------------------
BACKEND = "hf_tei"   # "hf_tei" (TEI /rerank) or "cohere" (Cohere-style /v1/rerank)
MODEL = ""           # e.g. "BAAI/bge-reranker-base" (TEI) or "BAAI/bge-reranker-v2-m3" (vLLM pooling)
URL = ""             # e.g. "http://localhost:8080"

ENDPOINT_TYPE = {"hf_tei": "hf_tei_rankings", "cohere": "cohere_rankings"}[BACKEND]

# ---- Workload ----------------------------------------------------------
CONCURRENCY = 1
REQUEST_COUNT = 10
RANKINGS_PASSAGES_MEAN = 5        # passages per request — set to your retriever's top-k
RANKINGS_PASSAGES_STDDEV = 1
PASSAGE_TOKENS_MEAN = 32
PASSAGE_TOKENS_STDDEV = 8
QUERY_TOKENS_MEAN = 16
QUERY_TOKENS_STDDEV = 4

# Relative to REPO_ROOT (aiperf runs with cwd=REPO_ROOT) — keep artifacts under notebooks/.
OUTPUT_DIR = "notebooks/artifacts/aiperf-uc8-rankings"


## 4. Test run — synthetic

AIPerf samples a passage count per request from `RANKINGS_PASSAGES_MEAN/STDDEV` and generates query/passage texts at the configured token lengths. As in UC7, the synthetic and custom runs write to separate `-synthetic` / `-custom` artifact directories.

In [ ]:
assert MODEL and URL, "Set MODEL and URL in the config cell above."

cmd = f"""
aiperf profile
    --model {MODEL}
    --url {URL}
    --endpoint-type {ENDPOINT_TYPE}
    --concurrency {CONCURRENCY}
    --request-count {REQUEST_COUNT}
    --rankings-passages-mean {RANKINGS_PASSAGES_MEAN}
    --rankings-passages-stddev {RANKINGS_PASSAGES_STDDEV}
    --rankings-passages-prompt-token-mean {PASSAGE_TOKENS_MEAN}
    --rankings-passages-prompt-token-stddev {PASSAGE_TOKENS_STDDEV}
    --rankings-query-prompt-token-mean {QUERY_TOKENS_MEAN}
    --rankings-query-prompt-token-stddev {QUERY_TOKENS_STDDEV}
    --artifact-dir {OUTPUT_DIR}-synthetic
""".split()

print("$ " + " ".join(cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
print(f"\nexit code: {result.returncode}")


## 5. Test run — custom input file

To replay real query/passages pairs, provide a JSONL file where each line names a `query` and its `passages`:

```json
{"texts": [{"name": "query", "contents": ["What is AI topic 0?"]}, {"name": "passages", "contents": ["AI passage 0"]}]}
```

The same file format works for both endpoint types. The cell below generates placeholder records in the tutorial's format — replace `records` with your production queries and candidate passages (e.g. dumped from your real retriever's top-k output).

In [ ]:
assert MODEL and URL, "Set MODEL and URL in the config cell above."

NUM_RECORDS = 10
records = [
    {
        "texts": [
            {"name": "query", "contents": [f"What is AI topic {i}?"]},
            {"name": "passages", "contents": [f"AI passage {i}"]},
        ]
    }
    for i in range(NUM_RECORDS)
]

INPUT_FILE = f"{OUTPUT_DIR}-inputs.jsonl"  # relative to REPO_ROOT, like OUTPUT_DIR
input_path = REPO_ROOT / INPUT_FILE
input_path.parent.mkdir(parents=True, exist_ok=True)
input_path.write_text("\n".join(json.dumps(r) for r in records) + "\n")
print(f"Wrote {len(records)} records to {input_path}")

cmd = f"""
aiperf profile
    --model {MODEL}
    --url {URL}
    --endpoint-type {ENDPOINT_TYPE}
    --input-file {INPUT_FILE}
    --custom-dataset-type single_turn
    --request-count {len(records)}
    --artifact-dir {OUTPUT_DIR}-custom
""".split()

print("$ " + " ".join(cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
print(f"\nexit code: {result.returncode}")


## 6. Results analysis

Same exports as every AIPerf run (see the UC2 notebook for the full tour of the export files). For rankings, the per-request statistics section carries **request latency**; the run-totals section carries **request throughput**.

In [ ]:
from io import StringIO

import pandas as pd


def read_export(path):
    """Split the multi-section AIPerf CSV into (stats, totals) DataFrames.

    Section 1: per-request statistics (Metric, avg, min, max, p50, ..., std).
    Section 2: single-value run totals (Metric, Value).
    Section 3 (GPU telemetry, if present) is ignored here.
    """
    sections = Path(path).read_text().strip().split("\n\n")
    stats = pd.read_csv(StringIO(sections[0]))
    totals = (pd.read_csv(StringIO(sections[1]))
              if len(sections) > 1 else pd.DataFrame(columns=["Metric", "Value"]))
    return stats, totals


def stat(stats, pattern, col="avg"):
    m = stats[stats["Metric"].str.lower().str.contains(pattern)]
    return float(m[col].iloc[0]) if not m.empty else None


rows, full = [], {}
for run in ("synthetic", "custom"):
    d = REPO_ROOT / f"{OUTPUT_DIR}-{run}"
    csv_path = next(d.rglob("profile_export_aiperf.csv"), None)
    if csv_path is None:
        print(f"No profile_export_aiperf.csv under {d} — skipping the '{run}' run.")
        continue
    stats, totals = read_export(csv_path)
    full[run] = stats
    thr = totals[totals["Metric"].str.lower().str.contains("request throughput")]
    rows.append({
        "run": run,
        "request_latency_avg_ms": stat(stats, "request latency"),
        "request_latency_p50_ms": stat(stats, "request latency", "p50"),
        "request_latency_p99_ms": stat(stats, "request latency", "p99"),
        "input_seq_len_avg": stat(stats, "input sequence length"),
        "request_throughput_rps": float(thr["Value"].iloc[0]) if not thr.empty else None,
    })

display(pd.DataFrame(rows))

# Full per-request statistics for each completed run:
for run, stats in full.items():
    print(f"=== {run} run — full statistics ===")
    display(stats)


### Notes

- **Passage count dominates cost.** Latency grows with `RANKINGS_PASSAGES_MEAN`, since every query–passage pair is scored. Profile at your retriever's real top-k, and try a lower k if the latency budget is tight.
- To compare serving stacks for the same reranker family (TEI vs. vLLM pooling), flip `BACKEND`/`MODEL`/`URL` and keep the workload knobs identical.
- If you hit a flag-validation error, check that you haven't mixed the `--rankings-*` token flags with the `--prompt-input-tokens-*` family — they are mutually exclusive.